In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import tensorflow as tf
import types
import pandas as pd
from dataclasses import dataclass
from typing import Any, List, Optional, Sequence, Tuple

def change_to_repo_root(marker: str = "src") -> None:
    """Change CWD to the repository root (parent of `src`)."""
    here = Path.cwd()
    for parent in [here] + list(here.parents):
        if (parent / marker).is_dir():
            os.chdir(parent)
            break

change_to_repo_root("WIP")
ROOT = Path.cwd()
WIP_SRC = ROOT / "WIP" / "src"
CORE_SRC = ROOT / "src"
WIP= ROOT / "WIP"
sys.path.insert(0, str(WIP_SRC))
sys.path.insert(0, str(CORE_SRC))
sys.path.insert(0, str(WIP))

from Q_Sea_Battle_New.pyr_dataset_conversion_utilities import  convert_all_traces
from Q_Sea_Battle_New.pyr_dataset_generation_utilities import generate_pyr_dataset
from Q_Sea_Battle_New.pyr_internal_model_b import PyrInternalModelB
from Q_Sea_Battle_New.pyr_combine_layer_b import PyrCombineLayerB
from Q_Sea_Battle.game_layout import GameLayout


In [2]:

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------
FIELD_SIZE = 4          # 4x4 -> N=16 (requires N a power of 2)
N2 = FIELD_SIZE * FIELD_SIZE
COMMS_SIZE = 1          # Pyramid requires 1 comm bit
P_HIGH = 1.0            # PR-assisted correlation parameter (stochastic mode, 2nd measurement)
GAMES_IN_EVAL_TOURNAMENT = 1000
NUM_GAMES_DATASET = 15_000
SEED = 1234
BETA_FIRST_MEASUREMENT = 10.0
BETA_TEACHER = 1.0
BATCH = 256
MAX_EPOCHS = 250

LAYOUT = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
DEPTH = int(np.log2(N2))  # number of levels in the pyramid (log2 of number of cells)



In [3]:
# --------------------------
# Generate canonical dataset (bits)
# --------------------------
ds_bits = generate_pyr_dataset(n2=N2, num_games=NUM_GAMES_DATASET, seed=SEED, validate=True)

In [4]:
tr = convert_all_traces(
    ds_bits,
    rep_field="hard_logit",        
    rep_gun="hard_logit",
    rep_comms="hard_logit",
    rep_meas_in_a="hard_logit",
    rep_meas_out_a="hard_logit",
    rep_meas_in_b="hard_logit",
    rep_meas_out_b="hard_logit",
    rep_shoot="hard_logit",
    beta=BETA_TEACHER,
)

# Model-B inputs (level-0 gun + comm0, plus A's previous lists)
gun_logits_np   = tr["gun"][0]      # (N, n2)
comm0_logits_np = tr["comms"][0]    # (N, 1)

prev_meas_list_np = [tr["meas_in_a"][d]  for d in range(DEPTH)]   # list of (N, k_d)
prev_out_list_np  = [tr["meas_out_a"][d] for d in range(DEPTH)]   # list of (N, k_d)

# Model-B targets (what you compare meas_b/out_b against)
#meas_b_list_np = [tr["meas_in_b"][d]  for d in range(DEPTH)]      # list of (N, k_d)
#out_b_list_np  = [tr["meas_out_b"][d] for d in range(DEPTH)]      # list of (N, k_d)

# Gun and comms targets are not used for training, but we convert them anyway for potential diagnostics
#gun_logits_list_np = [tr["gun"][d] for d in range(DEPTH+1)]      # list of (N, n2)
comm_logits_list_np = [tr["comms"][d] for d in range(DEPTH+1)]

# Shoot target (what you compare shoot_logit against)
#shoot_target_np = tr["shoot"]          # (N, 1) bits


# numpy -> tf
gun_logits   = tf.constant(gun_logits_np, tf.float32)         # (N,n2) logits
comm0_logits = tf.constant(comm0_logits_np, tf.float32)       # (N,1)  logits

prev_meas_t  = tuple(tf.constant(a, tf.float32) for a in prev_meas_list_np)  # tuple of (N,k_d) logits
prev_out_t   = tuple(tf.constant(a, tf.float32) for a in prev_out_list_np)   # tuple of (N,k_d) logits

#meas_tgt_t   = tuple(tf.constant(a, tf.float32) for a in meas_b_list_np)     # tuple of (N,k_d) logits (TARGET)
#out_tgt_t    = tuple(tf.constant(a, tf.float32) for a in out_b_list_np)      # tuple of (N,k_d) logits (TARGET)

#gun_tgt_list_t = tuple(tf.constant(a, tf.float32) for a in gun_logits_list_np)   # tuple of (N,n2) logits (TARGET)
comm_tgt_list_t = tuple(tf.constant(a, tf.float32) for a in comm_logits_list_np)   # tuple of (N,1) logits (TARGET)

#shoot_bits   = tf.constant(shoot_target_np, tf.float32)       # (N,1) bits

X_train = (gun_logits, comm0_logits, prev_meas_t, prev_out_t, )
Y_train = (comm_tgt_list_t,)

tfds = tf.data.Dataset.from_tensor_slices((X_train, Y_train))
tfds = tfds.shuffle(200_000, seed=SEED, reshuffle_each_iteration=True)
tfds = tfds.batch(BATCH).prefetch(tf.data.AUTOTUNE)


In [5]:
def bit_acc(y_bits, logits):
    pred = tf.cast(logits >= 0.0, tf.float32)
    return tf.reduce_mean(tf.cast(tf.equal(pred, y_bits), tf.float32))

In [6]:
@tf.function
def comm_bce_loss_weighted(comms_list, comm_tgt_list, depth_mask):
    """
    comms_list: Python list length depth+1 of (B,1) logits
    comm_tgt_list: list length depth+1 of (B,1) logits
    depth_mask: tf.Tensor shape (depth,), weights for d=1..depth (float32)

    Returns:
      loss_scalar, per_depth_loss (depth,)
    """
    # Stack d=1..depth into (B, depth, 1)
    pred = tf.stack([tf.cast(x, tf.float32) for x in comms_list[1:]], axis=1)
    tgt  = tf.stack([tf.cast(x, tf.float32) for x in comm_tgt_list[1:]], axis=1)

    # Target bits from target logits sign
    tgt_bits = tf.cast(tgt >= 0.0, tf.float32)

    # BCE per example per depth: (B, depth, 1)
    per_ex = tf.nn.sigmoid_cross_entropy_with_logits(labels=tgt_bits, logits=pred)

    # Mean over batch and last dim -> per-depth scalar: (depth,)
    per_depth = tf.reduce_mean(per_ex, axis=[0, 2])

    # Apply mask and normalize by active weight sum (scale-stable across schedules)
    depth_mask = tf.cast(depth_mask, tf.float32)
    num = tf.reduce_sum(per_depth * depth_mask)
    den = tf.maximum(tf.reduce_sum(depth_mask), 1e-8)
    return num / den, per_depth
@tf.function
def train_step(
    model,
    opt,
    x_gun, x_comm, x_prev_meas, x_prev_out,
    comm_tgt_list,
    depth_mask,
):
    with tf.GradientTape() as tape:
        shoot_logit, meas_b_list, out_b_list, comms_list, guns_list = model.compute_with_internal(
            x_gun, x_comm, x_prev_meas, x_prev_out, harden_between_levels= False)

        comm_loss, per_depth = comm_bce_loss_weighted(comms_list, comm_tgt_list, depth_mask)

        # --- fast "last comm bit-accuracy" (target vs model) ---
        # target bit from target logit sign
        tgt_last_bits = tf.cast(tf.cast(comm_tgt_list[-1], tf.float32) >= 0.0, tf.float32)  # (B,1)
        pred_last_logit = tf.cast(comms_list[-1], tf.float32)                               # (B,1)

        # bit_acc expects labels as {0,1} and logits as real
        comm_last_acc = bit_acc(tgt_last_bits, pred_last_logit)

    vars_ = model.trainable_variables
    grads = tape.gradient(comm_loss, vars_)
    opt.apply_gradients([(g, v) for g, v in zip(grads, vars_) if g is not None])

    return comm_loss, per_depth, comm_last_acc
SEED = 6532
# Speed-optimized training cell 
import tensorflow as tf
import gc, types

tf.keras.backend.clear_session()
gc.collect()

# Do not force eager
tf.config.run_functions_eagerly(False)

# --------------------------
# Model: instantiate + patch compute_with_internal
# --------------------------
model_b = PyrInternalModelB(
    LAYOUT,
    sr_mode="replay",
    alpha = 5.0,
    seed=SEED
)


opt = tf.keras.optimizers.Adam(1e-3)

epoch = 0
while True:
    epoch += 1

    if epoch <= 7:
        depth_mask =  tf.constant([1.0, 0.0, 0.0, 0.0], tf.float32)   # only comm[1]
    elif epoch <= 12:
        depth_mask = tf.constant([1.0, 1.0, 0.0, 0.0], tf.float32)   # comm[1..2]
    elif epoch <= 17:
        depth_mask = tf.constant([1.0, 1.0, 1.0, 0.0], tf.float32)   # comm[1..3]
    else:
        depth_mask = tf.constant([1.0, 1.0, 1.0, 1.0], tf.float32)   # only last comm (shoot)

    m_loss = tf.keras.metrics.Mean()
    last_per_depth = None
    last_acc = None

    for (x_gun, x_comm, x_prev_meas, x_prev_out), (y_comm_tgt_list,) in tfds:
        loss, per_depth, acc = train_step(
            model_b, opt,
            x_gun, x_comm, x_prev_meas, x_prev_out,
            y_comm_tgt_list,
            depth_mask
        )
        m_loss.update_state(loss)
        last_per_depth = per_depth
        last_acc = acc

    # per_depth is for d=1..depth in order; print nicely
    pdn = last_per_depth.numpy() if last_per_depth is not None else None
    print(f"Epoch {epoch:02d} last_acc={last_acc.numpy():.4f} comm_loss={m_loss.result().numpy():.4f}  mask={depth_mask.numpy().tolist()}  per_depth={['{:.4f}'.format(x) for x in pdn]}  ")

    if last_acc > 0.9999 or epoch >= MAX_EPOCHS:
        print("Early stopping at 100% last comm accuracy.")
        break
    
model_weights_dir=Path("WIP/weights_pyr_models")
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
filename_template=f"model_b_weights_{timestamp}.weights.h5"
model_b.save_weights_to(model_weights_dir / filename_template)
print(f"Model weights saved to {model_weights_dir / filename_template}")


Epoch 01 last_acc=0.4408 comm_loss=0.6955  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.6836', '0.7120', '0.7087', '0.6971']  
Epoch 02 last_acc=0.4934 comm_loss=0.6790  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.6614', '0.6902', '0.6883', '0.6932']  
Epoch 03 last_acc=0.5066 comm_loss=0.6624  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.6471', '0.6888', '0.7068', '0.6933']  
Epoch 04 last_acc=0.5197 comm_loss=0.6396  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.5937', '0.6943', '0.6955', '0.6944']  
Epoch 05 last_acc=0.5395 comm_loss=0.6120  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.5945', '0.6829', '0.6935', '0.6921']  
Epoch 06 last_acc=0.4605 comm_loss=0.5822  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.5539', '0.7122', '0.6981', '0.6963']  
Epoch 07 last_acc=0.4605 comm_loss=0.5534  mask=[1.0, 0.0, 0.0, 0.0]  per_depth=['0.5602', '0.7092', '0.6900', '0.6949']  
Epoch 08 last_acc=0.4868 comm_loss=0.6086  mask=[1.0, 1.0, 0.0, 0.0]  per_depth=['0.5203', '0.6855', '0.7032', '0.6944']  
Epoch 09 last_a